## Improving the ML pipeline in baseline models

#### Importing Libraries

In [14]:
import pandas as pd
import numpy as np
import category_encoders as ce

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

#### Load data

In [15]:
train = pd.read_csv("./../data/processed/adult_train.csv")
test = pd.read_csv("./../data/processed/adult_test.csv")

train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Data Preprocessing

#### Cleaning missing values

In [16]:
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)

train.dropna(inplace=True)
test.dropna(inplace=True)


#### Encoding target

In [17]:
train["income"] = train["income"].str.strip().map({"<=50K": 0, ">50K": 1})
test["income"] = test["income"].str.strip().map({"<=50K": 0, ">50K": 1})

#### Save protected attributes (needed for fairness metrics later)

In [18]:
# Must be saved BEFORE one-hot encoding explodes these columns
sex_train = train["sex"].copy()
sex_test  = test["sex"].copy()
race_train = train["race"].copy()
race_test  = test["race"].copy()

#### Encoding Features

In [19]:
#High-cardinality - Target Encoding
target_enc_cols = ["occupation", "native-country"]
te = ce.TargetEncoder(cols=target_enc_cols)
train[target_enc_cols] = te.fit_transform(train[target_enc_cols], train["income"])
test[target_enc_cols] = te.transform(test[target_enc_cols])

#Low-cardinality - One-Hot Encoding
ohe_cols = ["workclass", "marital-status", "relationship", "race", "sex"]
train = pd.get_dummies(train, columns=ohe_cols)
test = pd.get_dummies(test, columns=ohe_cols)

#Align columns so train/test have the same OHE columns
train, test = train.align(test, join="left", axis=1, fill_value=0)

print(f"Train shape after encoding: {train.shape}")
print(f"Test shape after encoding:  {test.shape}")

Train shape after encoding: (30162, 37)
Test shape after encoding:  (15060, 37)


### Feature Engineering

In [20]:
for df in [train, test]:
    #Bin age into groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 25, 45, 65, 100],
        labels=["young", "mid", "senior", "elder"]
    ).cat.codes  #encode as integers

    #Log1p transform for skewed capital columns
    df["capital-gain"] = np.log1p(df["capital-gain"])
    df["capital-loss"] = np.log1p(df["capital-loss"])

    #New feature - Net capital
    df["capital-net"] = df["capital-gain"] - df["capital-loss"]

print("New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net")

New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net


### Drop redundant features

In [21]:
#Verifying education vs education-num correlation before dropping
from sklearn.preprocessing import LabelEncoder
_edu_encoded = LabelEncoder().fit_transform(train["education"])
corr = np.corrcoef(_edu_encoded, train["education-num"])[0, 1]
print(f"Pearson correlation between education (label-encoded) and education-num: {corr:.4f}")


drop_cols = ["education", "fnlwgt"]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

Pearson correlation between education (label-encoded) and education-num: 0.3454
Dropped columns: ['education', 'fnlwgt']


### Split X and y

In [22]:
X_train = train.drop("income", axis=1)
X_test = test.drop("income", axis=1)

y_train = train["income"]
y_test = test["income"]

print(f"X_train shape:{X_train.shape}, X_test shape: {X_test.shape}")

X_train shape:(30162, 36), X_test shape: (15060, 36)


### Scale numerical features

In [23]:
num_cols = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week", "capital-net", "occupation", "native-country", "age_group"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## GBDT — Baseline vs Improved Pipeline

In [24]:
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)

y_pred = gbdt.predict(X_test)
y_proba = gbdt.predict_proba(X_test)[:, 1]

baseline = {"Accuracy": 0.8611, "F1": 0.6812, "AUC": 0.9181}
improved = {
    "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    "F1":       round(f1_score(y_test, y_pred), 4),
    "AUC":      round(roc_auc_score(y_test, y_proba), 4),
}

comparison = pd.DataFrame({"Baseline GBDT": baseline, "Improved GBDT": improved})
print(comparison)

for metric in ["Accuracy", "F1", "AUC"]:
    delta = improved[metric] - baseline[metric]
    direction = "▲" if delta >= 0 else "▼"
    print(f"{metric}: {direction} {abs(delta):.4f}")

          Baseline GBDT  Improved GBDT
Accuracy         0.8611         0.8647
F1               0.6812         0.6956
AUC              0.9181         0.9191
Accuracy: ▲ 0.0036
F1: ▲ 0.0144
AUC: ▲ 0.0010


---
## Fairness-Aware Learning

#### Task 1 — Baseline fairness metrics (Demographic Parity & Equalized Odds)

In [25]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

# Reuse predictions already made by the improved-pipeline GBDT
y_pred_baseline = gbdt.predict(X_test)

print("=== Baseline GBDT — Fairness Metrics ===\n")
print(f"{'Attribute':<8}  {'Demographic Parity Diff':>24}  {'Equalized Odds Diff':>20}")
print("-" * 58)
for attr_name, s_test_arr in [("sex", sex_test), ("race", race_test)]:
    dpd = demographic_parity_difference(y_test, y_pred_baseline, sensitive_features=s_test_arr)
    eod = equalized_odds_difference(y_test, y_pred_baseline, sensitive_features=s_test_arr)
    print(f"{attr_name:<8}  {dpd:>24.4f}  {eod:>20.4f}")

print("\nInterpretation:")
print("  DPD: max difference in P(ŷ=1) across groups  (0 = perfect parity)")
print("  EOD: max difference in TPR or FPR across groups  (0 = equalized odds)")

=== Baseline GBDT — Fairness Metrics ===

Attribute   Demographic Parity Diff   Equalized Odds Diff
----------------------------------------------------------
sex                         0.1817                0.1070
race                        0.1870                0.2194

Interpretation:
  DPD: max difference in P(ŷ=1) across groups  (0 = perfect parity)
  EOD: max difference in TPR or FPR across groups  (0 = equalized odds)


#### Task 2 — Sample reweighing (inverse group × label frequency)

In [26]:
# Build a composite key: sex_race_label  (one cell per combination)
group_label_key = (
    sex_train.reset_index(drop=True).astype(str) + "_"
    + race_train.reset_index(drop=True).astype(str) + "_"
    + y_train.reset_index(drop=True).astype(str)
)

# Frequency of each cell in training data
cell_freq = group_label_key.map(group_label_key.value_counts(normalize=True))

# Weight = 1/freq, then normalize so mean weight = 1 (keeps loss scale stable)
sample_weights = 1.0 / cell_freq
sample_weights = sample_weights * (len(sample_weights) / sample_weights.sum())

print("Sample weight summary:")
print(sample_weights.describe().round(4))
print(f"\nDistinct group×label cells: {group_label_key.nunique()}")

# Train reweighed GBDT — same hyperparams as baseline
gbdt_rw = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt_rw.fit(
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True),
    sample_weight=sample_weights.values,
)
print("\nReweighed GBDT trained.")

Sample weight summary:
count    30162.0000
mean         1.0000
std          6.2668
min          0.1239
25%          0.1239
50%          0.2178
75%          0.2570
max        377.0250
dtype: float64

Distinct group×label cells: 20

Reweighed GBDT trained.


#### Task 3 — Equalized odds post-processing (ThresholdOptimizer)

In [27]:
from fairlearn.postprocessing import ThresholdOptimizer

# Reset indices so fairlearn's internal alignment works cleanly
X_train_r = X_train.reset_index(drop=True)
y_train_r  = y_train.reset_index(drop=True)
X_test_r   = X_test.reset_index(drop=True)
y_test_r   = y_test.reset_index(drop=True)
sex_train_r = sex_train.reset_index(drop=True)
sex_test_r  = sex_test.reset_index(drop=True)

# ThresholdOptimizer wraps the already-fitted reweighed model (prefit=True)
# and learns per-group thresholds that satisfy equalized_odds on the training set.
# 'sex' is used as the sensitive attribute — binary, so threshold search is tractable.
to = ThresholdOptimizer(
    estimator=gbdt_rw,
    constraints="equalized_odds",   # equalise TPR AND FPR across sex groups
    objective="accuracy_score",     # among all threshold combinations, pick the one maximising accuracy
    prefit=True,
    predict_method="predict_proba",
)
to.fit(X_train_r, y_train_r, sensitive_features=sex_train_r)

y_pred_to = to.predict(X_test_r, sensitive_features=sex_test_r, random_state=42)
print("ThresholdOptimizer fitted and predictions generated.")

ThresholdOptimizer fitted and predictions generated.


#### Task 4 — Before vs. After: accuracy and fairness comparison

In [28]:
y_pred_rw = gbdt_rw.predict(X_test_r)

def fairness_row(label, y_true, y_pred, sex_arr, race_arr):
    return {
        "Model":        label,
        "Accuracy":     round(accuracy_score(y_true, y_pred), 4),
        "F1":           round(f1_score(y_true, y_pred), 4),
        "DPD (sex)":    round(demographic_parity_difference(y_true, y_pred, sensitive_features=sex_arr), 4),
        "EOD (sex)":    round(equalized_odds_difference(y_true, y_pred, sensitive_features=sex_arr), 4),
        "DPD (race)":   round(demographic_parity_difference(y_true, y_pred, sensitive_features=race_arr), 4),
        "EOD (race)":   round(equalized_odds_difference(y_true, y_pred, sensitive_features=race_arr), 4),
    }

rows = [
    fairness_row("Baseline GBDT",        y_test_r, y_pred_baseline,         sex_test_r, race_test.reset_index(drop=True)),
    fairness_row("Reweighed GBDT",        y_test_r, y_pred_rw,               sex_test_r, race_test.reset_index(drop=True)),
    fairness_row("ThresholdOptimizer",    y_test_r, y_pred_to,               sex_test_r, race_test.reset_index(drop=True)),
]

comparison_df = pd.DataFrame(rows).set_index("Model")
print(comparison_df.to_string())

print("\nNote: Lower |DPD| and |EOD| = fairer. ThresholdOptimizer optimises for sex;")
print("race fairness may improve as a side-effect but is not directly constrained.")

                    Accuracy      F1  DPD (sex)  EOD (sex)  DPD (race)  EOD (race)
Model                                                                             
Baseline GBDT         0.8647  0.6956     0.1817     0.1070      0.1870      0.2194
Reweighed GBDT        0.8102  0.6851     0.2102     0.1076      0.1635      0.1274
ThresholdOptimizer    0.8454  0.6422     0.1043     0.0161      0.1418      0.1284

Note: Lower |DPD| and |EOD| = fairer. ThresholdOptimizer optimises for sex;
race fairness may improve as a side-effect but is not directly constrained.
